# Prediction results

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import chi2
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import final_dataset, gold_final, models_dir

labels = ["afwijkend", "niet-afwijkend"]


def load_gold_preds(prefix):
    path = models_dir / f"{prefix}_gold_test_predictions.csv"
    if not path.exists():
        return None
    return pd.read_csv(path, encoding="utf-8-sig")


def metrics_from_preds(y_true, y_pred):
    y_true = pd.Series(y_true).astype(str)
    y_pred = pd.Series(y_pred).astype(str)
    n_afw = int((y_true == "afwijkend").sum())
    n_niet = int((y_true == "niet-afwijkend").sum())
    tp = int(((y_true == "afwijkend") & (y_pred == "afwijkend")).sum())
    tn = int(((y_true == "niet-afwijkend") & (y_pred == "niet-afwijkend")).sum())
    return {
        "n": int(len(y_true)),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0),
        "kappa": cohen_kappa_score(y_true, y_pred),
        "sensitivity": tp / n_afw if n_afw else np.nan,
        "specificity": tn / n_niet if n_niet else np.nan,
        "tp": tp, "tn": tn, "n_afw": n_afw, "n_niet": n_niet,
    }


def mcnemar(y_true, pred_a, pred_b):
    correct_a = np.array(pred_a) == np.array(y_true)
    correct_b = np.array(pred_b) == np.array(y_true)
    b = int((correct_a & ~correct_b).sum())
    c = int((~correct_a & correct_b).sum())
    if b + c == 0:
        return b, c, np.nan, np.nan
    stat = (abs(b - c) - 1) ** 2 / (b + c)
    p_value = 1 - chi2.cdf(stat, df=1)
    return b, c, stat, p_value


def load_all_predictions():
    classical = {
        "majority_baseline": "majority_baseline_minwords0",
        "tfidf_logreg": "tfidf_logreg_minwords0",
        "tfidf_mlp": "tfidf_mlp_minwords0",
        "tfidf_xgboost": "tfidf_xgboost_minwords0",
    }
    bert = {
        "bert_gp_only": models_dir / "bert" / "gp_only" / "gold_predictions.csv",
        "bert_gp_plus_radiology": models_dir / "bert" / "gp_plus_radiology" / "gold_predictions.csv",
    }
    predictions = {}
    for name, prefix in classical.items():
        df = load_gold_preds(prefix)
        if df is not None:
            predictions[name] = df
    for name, path in bert.items():
        if path.exists():
            df = pd.read_csv(path, encoding="utf-8-sig")
            if "case_id" not in df.columns:
                df = df.reset_index().rename(columns={"index": "case_id"})
            predictions[name] = df
    return predictions


preds = load_all_predictions()
pd.DataFrame({"loaded_models": list(preds)})

## Gold test performance

In [ ]:
def metric_value(metrics, key):
    value = metrics.get(key)
    if isinstance(value, dict):
        return value.get("value")
    return value


def read_json(path):
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))


def performance_row(model, y_true, y_pred, auc=np.nan):
    m = metrics_from_preds(y_true, y_pred)
    return {
        "model": model,
        "n": m["n"],
        "accuracy": m["accuracy"],
        "macro_f1": m["macro_f1"],
        "kappa": m["kappa"],
        "auc": auc,
        "sensitivity": m["sensitivity"],
        "specificity": m["specificity"],
        "tp": m["tp"], "tn": m["tn"], "n_afw": m["n_afw"], "n_niet": m["n_niet"],
    }


rows = []
for model, prefix in [
    ("Majority baseline", "majority_baseline_minwords0"),
    ("TF-IDF LogReg", "tfidf_logreg_minwords0"),
    ("TF-IDF MLP", "tfidf_mlp_minwords0"),
    ("TF-IDF XGBoost", "tfidf_xgboost_minwords0"),
]:
    df = load_gold_preds(prefix)
    metrics = read_json(models_dir / f"{prefix}_gold_test_metrics.json") or {}
    if df is not None:
        rows.append(performance_row(model, df["y_true"], df["y_pred"], auc=metric_value(metrics, "roc_auc")))

for model, path in [
    ("MedRoBERTa.nl-GP", models_dir / "bert" / "gp_only" / "gold_predictions.csv"),
    ("MedRoBERTa.nl-GP+report", models_dir / "bert" / "gp_plus_radiology" / "gold_predictions.csv"),
]:
    if path.exists():
        df = pd.read_csv(path, encoding="utf-8-sig")
        metrics = read_json(path.parent / "metrics.json") or {}
        rows.append(performance_row(model, df["y_true"], df["y_pred"], auc=metrics.get("roc_auc", np.nan)))

primary = pd.read_excel(final_dataset)
gold_source = pd.read_excel(gold_final).rename(columns={"definitief_handmatig_label": "handmatig_label"})
llm_gold = gold_source[["case_id", "handmatig_label"]].merge(
    primary[["case_id", "label_gecombineerd"]], on="case_id", how="left", validate="one_to_one"
)
llm_sub = llm_gold[llm_gold["handmatig_label"].isin(labels) & llm_gold["label_gecombineerd"].isin(labels)]
if len(llm_sub):
    rows.append(performance_row("LLM zero-shot combined", llm_sub["handmatig_label"], llm_sub["label_gecombineerd"], auc=np.nan))

performance_table = pd.DataFrame(rows).round(3)
performance_table[["model", "n", "accuracy", "macro_f1", "kappa", "auc", "sensitivity", "specificity"]]

## Top TF-IDF features

In [ ]:
path = models_dir / "tfidf_logreg_minwords0_top_words.csv"
if path.exists():
    top_words = pd.read_csv(path, encoding="utf-8-sig")
    top_words["feature"] = top_words["feature"].str.replace("tfidf__", "", regex=False).str.replace("manual__", "manual:", regex=False)
    top_words = top_words.sort_values("coefficient", ascending=False).groupby("class", as_index=False).head(20).reset_index(drop=True)
else:
    top_words = pd.DataFrame({"message": ["top-word file not found"]})
top_words

## McNemar

In [ ]:
names = list(preds)
rows = []
if len(names) >= 2:
    case_ids = set(preds[names[0]]["case_id"])
    for df in preds.values():
        case_ids &= set(df["case_id"])
    case_ids = sorted(case_ids)
    aligned = {name: df[df["case_id"].isin(case_ids)].set_index("case_id") for name, df in preds.items()}
    y_true = aligned[names[0]]["y_true"].loc[case_ids].tolist()
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            b_count, c_count, stat, p = mcnemar(y_true, aligned[a]["y_pred"].loc[case_ids], aligned[b]["y_pred"].loc[case_ids])
            rows.append({"model_a": a, "model_b": b, "b": b_count, "c": c_count, "chi2": stat, "p": p})
mcnemar_table = pd.DataFrame(rows).round(4)
mcnemar_table

## Selected McNemar tests

In [ ]:
selected_pairs = [
    ("majority_baseline", "tfidf_logreg"),
    ("majority_baseline", "tfidf_xgboost"),
    ("majority_baseline", "bert_gp_only"),
]
selected_mcnemar = mcnemar_table.merge(
    pd.DataFrame(selected_pairs, columns=["model_a", "model_b"]),
    on=["model_a", "model_b"],
    how="inner",
)
selected_mcnemar

## Word-count groups

In [ ]:
base_df = next((df for df in preds.values() if "word_count" in df.columns), None)
rows = []
if base_df is not None:
    groups = {
        "short (<10)": set(base_df[base_df["word_count"] < 10]["case_id"]),
        "long (>=10)": set(base_df[base_df["word_count"] >= 10]["case_id"]),
    }
    for group, ids in groups.items():
        for name, df in preds.items():
            sub = df[df["case_id"].isin(ids)]
            if len(sub):
                rows.append({"group": group, "model": name, **metrics_from_preds(sub["y_true"], sub["y_pred"])})
word_count_table = pd.DataFrame(rows).round(3)
word_count_table

## Word-count F1 difference

In [ ]:
model_names = {
    "majority_baseline": "Majority baseline",
    "tfidf_logreg": "TF-IDF LogReg",
    "tfidf_mlp": "TF-IDF MLP",
    "tfidf_xgboost": "TF-IDF XGBoost",
    "bert_gp_only": "MedRoBERTa.nl-GP",
}
if word_count_table.empty:
    word_count_delta_table = pd.DataFrame()
else:
    f1_pivot = word_count_table.pivot(index="model", columns="group", values="macro_f1")
    rows = []
    for key, name in model_names.items():
        if key in f1_pivot.index:
            short = f1_pivot.loc[key].get("short (<10)", np.nan)
            long = f1_pivot.loc[key].get("long (>=10)", np.nan)
            rows.append({"model": name, "short_f1": short, "long_f1": long, "delta_f1": long - short})
    word_count_delta_table = pd.DataFrame(rows).round(3)
word_count_delta_table

## min_words=10

In [ ]:
models = {
    "majority baseline": ("majority_baseline_minwords0", "majority_baseline_minwords10"),
    "tfidf logreg": ("tfidf_logreg_minwords0", "tfidf_logreg_minwords10"),
    "tfidf MLP": ("tfidf_mlp_minwords0", "tfidf_mlp_minwords10"),
    "tfidf XGBoost": ("tfidf_xgboost_minwords0", "tfidf_xgboost_minwords10"),
}
rows = []
for model, (prefix0, prefix10) in models.items():
    for filter_name, prefix in [("mw=0", prefix0), ("mw=10", prefix10)]:
        df = load_gold_preds(prefix)
        if df is not None:
            rows.append({"model": model, "filter": filter_name, **metrics_from_preds(df["y_true"], df["y_pred"])})
min_words_table = pd.DataFrame(rows).round(3)
min_words_table

## min_words=10 summary

In [ ]:
if min_words_table.empty:
    min_words_summary = pd.DataFrame()
else:
    base = min_words_table[min_words_table["filter"] == "mw=0"][["model", "macro_f1", "kappa", "specificity"]].rename(
        columns={"macro_f1": "macro_f1_mw0", "kappa": "kappa_mw0", "specificity": "specificity_mw0"}
    )
    long = min_words_table[min_words_table["filter"] == "mw=10"][["model", "n", "macro_f1", "kappa", "specificity"]].rename(
        columns={"n": "n_mw10", "macro_f1": "macro_f1_mw10", "kappa": "kappa_mw10", "specificity": "specificity_mw10"}
    )
    min_words_summary = base.merge(long, on="model", how="inner")
    min_words_summary["delta_f1"] = min_words_summary["macro_f1_mw10"] - min_words_summary["macro_f1_mw0"]
    min_words_summary = min_words_summary.round(3)
min_words_summary

## min_words=10 split sizes

In [ ]:
split_path = models_dir / "tfidf_logreg_minwords10_split_label_distribution.csv"
if split_path.exists():
    split_counts = pd.read_csv(split_path, encoding="utf-8-sig")
    min_words_split_sizes = split_counts.groupby("split", as_index=False)["count"].sum()
else:
    min_words_split_sizes = pd.DataFrame({"message": ["split distribution file not found"]})
min_words_split_sizes

## Threshold tuning

In [ ]:
tuning_paths = [
    models_dir / "test_tuning" / "tuning_results.json",
    models_dir.parent / "test_tuning" / "tuning_results.json",
]
tuning_path = next((path for path in tuning_paths if path.exists()), None)
if tuning_path is None:
    tuning_table = pd.DataFrame({"message": ["tuning result file not found"]})
else:
    tuning = pd.read_json(tuning_path)
    keep = [
        "model", "threshold", "n_test", "accuracy", "macro_f1", "kappa",
        "sensitivity_afwijkend", "specificity_niet_afwijkend", "tp", "fn", "fp", "tn",
    ]
    tuning_table = tuning[[c for c in keep if c in tuning.columns]].round(4)
tuning_table

## Best tuned configurations

In [ ]:
best_names = [
    "logreg balanced + threshold (thr=0.46)",
    "mlp balanced + threshold (thr=0.48)",
    "xgboost spw=1.66 (retrained)",
]
if "model" not in tuning_table.columns:
    best_tuned_table = pd.DataFrame()
else:
    best_tuned_table = tuning_table[tuning_table["model"].isin(best_names)].copy()
    best_tuned_table["model"] = pd.Categorical(best_tuned_table["model"], categories=best_names, ordered=True)
    best_tuned_table = best_tuned_table.sort_values("model")
    best_tuned_table = best_tuned_table[["model", "threshold", "macro_f1", "kappa", "sensitivity_afwijkend", "specificity_niet_afwijkend"]]
best_tuned_table